# LingBot-Map on a big GPU — upstream's own demo scenes, end to end

**What this answers:** is our 8 GB RTX 4060 Ti the reason upstream's `example/courthouse`
reconstructs into a scribble here, when their published demo of the same frames is coherent?

Measured locally (`notes/open-questions.md`): the KV cache that bounds streaming VRAM is also the
model's memory horizon. Upstream's `demo.py` defaults to `kv_cache_sliding_window=64`. This box
tops out at **~24** — 64 and 48 both OOM, and halving the sequence with `--stride 2` does not buy
a bigger cache, so the ceiling is set by cache size, not sequence length. At the 16 we can afford,
`traj_length_over_extent` is **24.9** against `loop`'s healthy 3.36.

So either a bigger card fixes courthouse outright, or the cache is not the variable and something
else is wrong. This notebook settles it by running **the exact upstream config** on a card that
can hold it, and by sweeping the cache to find where the cliff actually is.

## What it does

1. Reports the GPU it landed on.
2. Installs LingBot-Map, fetches the checkpoint from HuggingFace.
3. Runs **both** demo scenes — `example/loop` and `example/courthouse` — at upstream's settings
   (`streaming`, `kvsw 64`, `keyframe_interval 1`, `num_scale_frames 8`, 518 px, crop,
   `--mask_sky` on courthouse only, exactly as upstream's README invokes them).
4. Sweeps `kv_cache_sliding_window` over 16 → 128 on courthouse, including a
   `kvsw 16 / keyframe_interval 2` variant that holds the *temporal span* fixed while halving the
   *view count* — that pair is what separates "the model needs more cached views" from
   "the model needs a longer horizon".
5. Cleans both clouds hard in Open3D — statistical outlier removal, radius filtering for the
   flying-pixel streaks, voxel downsample, camera-height scale calibration, ground alignment.
6. Prints a verdict: **is compute the bottleneck**, and should Phase 3 inference move to Colab.

## What you get back

A zip (and optionally a Drive folder) per scene containing `cloud_clean.ply` in **metres**,
`run.json`, `scale.json`, `clean_stats.json`, renders, and the raw log — plus ready-to-paste
`notes/experiments.md` rows.

> **Runtime → Change runtime type → A100 or L4 before running anything.** On a T4 (16 GB, Turing)
> the model runs in fp16 instead of bf16 and `kvsw 128` will not fit; the notebook says so and
> keeps going.

**Nothing here is a fork of the pipeline.** It runs `recon/reconstruct.py`, `calibrate_scale.py`
and `clean_cloud.py` from the repo unmodified — same code, same flags, different card. That is the
only way the comparison means anything.

## 1 · Which GPU did we get?

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU (A100 or L4)."
P = torch.cuda.get_device_properties(0)
VRAM_GB = P.total_memory / 1e9
CAP = torch.cuda.get_device_capability()
print(f"\n{P.name}  {VRAM_GB:.1f} GB  sm_{CAP[0]}{CAP[1]}  torch {torch.__version__}")
print(f"local baseline for comparison: RTX 4060 Ti, 8.6 GB, sm_89  ->  {VRAM_GB/8.6:.1f}x the VRAM")

# reconstruct.py picks bf16 on sm_80+ and fp16 below it. A T4 therefore runs a
# *different numeric path* than our box does, which muddies the comparison; an
# A100/L4 keeps bf16 and changes only the one variable we care about.
if CAP[0] < 8:
    print("\nWARNING: pre-Ampere card -> fp16 instead of bf16. Two variables change at once.")
if VRAM_GB < 20:
    print("WARNING: <20 GB. kvsw 128 will probably OOM; the rest of the ladder should fit.")

## 2 · Configuration

Defaults reproduce upstream's README commands exactly. The only deliberate departures are
export-side (they change which points land in the `.ply`, not the geometry the model computes):
`--conf_percentile 55` instead of upstream's absolute `--conf_threshold 1.5`, and
`--pixel_stride 2` for density.

`keyframe_interval` is pinned to **1** rather than left on auto. Upstream's auto is
`(n_frames + 319) // 320` = 1 for both scenes; ours is `ceil(n_frames / 240)`, which is 1 for
loop's 237 frames but **2** for courthouse's 286. Leaving it on auto would have silently halved
courthouse's keyframe density relative to upstream and quietly invalidated the comparison.

In [ ]:
# ── scenes ───────────────────────────────────────────────────────────────────
SCENES        = ["loop", "courthouse"]   # upstream also ships "university" (324 frames)
CHECKPOINT    = "lingbot-map.pt"         # paper/benchmark/demo checkpoint. "lingbot-map-long.pt"
                                         # is the long-sequence variant that OOMs locally at 286
                                         # frames -- worth a try here once the main runs land.

# ── upstream demo.py defaults, verbatim ──────────────────────────────────────
UPSTREAM = dict(
    mode="streaming",
    kv_cache_sliding_window=64,
    keyframe_interval=1,
    num_scale_frames=8,
    camera_num_iterations=4,
    image_size=518,
    preprocess_mode="crop",   # upstream's demo.load_images hardcodes crop
)
MASK_SKY = {"courthouse": True, "university": True, "loop": False}  # per upstream's README

# ── export knobs (post-inference; cannot affect poses or drift) ──────────────
CONF_PERCENTILE = 55
PIXEL_STRIDE    = 2          # 1 = ~4x the points, ~4x the PLY. 2 matches our local runs.
VRAM_FRACTION   = 0.92       # no WSL driver fragility here, so higher than our local 0.85

# ── the cache sweep: (kv_cache_sliding_window, keyframe_interval) ───────────
# 16 reproduces our local courthouse run exactly -> a controlled A/B where the card is the
# only variable. (16, 2) doubles the horizon at the same view count; (32, 1) holds the horizon
# and doubles the count. Compare those two and you know which quantity the model is short of.
RUN_SWEEP    = True
SWEEP_SCENE  = "courthouse"
SWEEP        = [(16, 1), (16, 2), (24, 1), (32, 1), (64, 1), (128, 1)]

# ── cleanup ──────────────────────────────────────────────────────────────────
CLEAN_HEAVY  = True          # tighter than repo defaults: std-ratio 2.0->1.5, min-nb 12->16
CAMERA_HEIGHT_M = 1.5        # assumed eye height for the scale anchor

# ── output ───────────────────────────────────────────────────────────────────
SAVE_TO_DRIVE = False        # mount Drive and copy results there as well as zipping
KEEP_RAW_PLY  = False        # raw cloud.ply is 100-400 MB per scene; the clean one is the artifact

# ── optional: your own footage (leave empty for the demo scenes only) ───────
VIDEOS = {}                  # {"my_trail": "/content/drive/MyDrive/trail.MOV"}
VIDEO_FPS = 5

WORK = "/content/gd"
print("config loaded")

## 3 · Install

`lingbot-map`'s `pyproject.toml` does **not** pin torch, so this installs on top of whatever
torch Colab ships and leaves the CUDA stack alone. If pip asks you to restart the runtime, do it
and re-run from cell 1 — the clone and any downloads are already on disk and will be skipped.

In [ ]:
import os, pathlib, subprocess, sys

LINGBOT_SRC = "/content/lingbot-map"
os.environ["LINGBOT_SRC"] = LINGBOT_SRC   # recon/reconstruct.py reads this to import upstream demo.py

if not pathlib.Path(LINGBOT_SRC, ".git").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git {LINGBOT_SRC}
else:
    print("lingbot-map already cloned")

# [vis] pulls viser/trimesh/matplotlib/onnxruntime. onnxruntime is what --mask_sky needs.
# `!pip` rather than `%pip` because the line magic does not expand {LINGBOT_SRC}.
!pip install -q -e "{LINGBOT_SRC}[vis]"
!pip install -q open3d imageio-ffmpeg

import open3d as o3d
print("open3d", o3d.__version__)
print("frames on disk:", {d.name: len(list(d.glob('*.png')))
                          for d in sorted(pathlib.Path(LINGBOT_SRC, "example").iterdir()) if d.is_dir()})

## 4 · Get the GeologicDome `recon/` scripts

The repo is private, so there are three ways in and the cell tries them in order. The zip upload
is the zero-setup one — make it locally with:

```powershell
Compress-Archive -Path recon -DestinationPath recon.zip -Force
```

`recon/` is ~80 KB. Nothing else from the repo is needed: `reconstruct.py` imports upstream's
`demo.py` (via `LINGBOT_SRC`) and `calibrate_scale.py` imports `clean_cloud.py` from beside it.

In [ ]:
import pathlib, shutil, zipfile

RECON = pathlib.Path("/content/recon")
NEEDED = ["reconstruct.py", "calibrate_scale.py", "clean_cloud.py", "inspect_cloud.py",
          "extract_frames.py"]


def _ok(d):
    return d.is_dir() and all((d / n).exists() for n in NEEDED)


if not _ok(RECON):
    # 1) already in a mounted Drive
    for c in ["/content/drive/MyDrive/GeologicDome/recon", "/content/drive/MyDrive/recon"]:
        if _ok(pathlib.Path(c)):
            shutil.copytree(c, RECON, dirs_exist_ok=True)
            print("copied from Drive:", c)
            break

if not _ok(RECON):
    # 2) private clone with a token stored in Colab Secrets as GD_TOKEN
    try:
        from google.colab import userdata
        tok = userdata.get("GD_TOKEN")
        url = f"https://{tok}@github.com/adikothuri3/geologic_dome_sim_onboarding.git"
        subprocess.run(["git", "clone", "--depth", "1", url, "/content/gd_repo"], check=True)
        shutil.copytree("/content/gd_repo/recon", RECON, dirs_exist_ok=True)
        print("cloned private repo")
    except Exception as e:
        print("no token clone:", type(e).__name__)

if not _ok(RECON):
    # 3) upload recon.zip
    from google.colab import files
    print("Upload recon.zip  (PowerShell: Compress-Archive -Path recon -DestinationPath recon.zip -Force)")
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content/_up")
    src = next(p.parent for p in pathlib.Path("/content/_up").rglob("reconstruct.py"))
    shutil.copytree(src, RECON, dirs_exist_ok=True)

assert _ok(RECON), f"missing {[n for n in NEEDED if not (RECON / n).exists()]}"
print("pipeline code ready:", sorted(p.name for p in RECON.glob('*.py')))

## 5 · Checkpoint + sky segmentation

4.63 GB from HuggingFace, ~2 min on Colab's backbone. `skyseg.onnx` is only needed for courthouse
(upstream runs that scene with `--mask_sky`); it runs on CPU by design, since upstream wires sky
masking into its *viewer* only and our `.ply` export needed its own path.

In [ ]:
import pathlib
from huggingface_hub import hf_hub_download

CKPT_DIR = pathlib.Path("/content/ckpt"); CKPT_DIR.mkdir(exist_ok=True)

CKPT = pathlib.Path(hf_hub_download("robbyant/lingbot-map", CHECKPOINT,
                                    local_dir=str(CKPT_DIR)))
print(f"{CKPT}  {CKPT.stat().st_size/1e9:.2f} GB")

SKYSEG = CKPT_DIR / "skyseg.onnx"
if any(MASK_SKY.get(s) for s in SCENES) and not SKYSEG.exists():
    # Same file upstream's viewer pulls on first use; a 176 MB U-Net that our
    # reconstruct.py runs on CPU so it never competes with the aggregator for VRAM.
    !curl -sL -o {SKYSEG} https://huggingface.co/JianyuanWang/skyseg/resolve/main/skyseg.onnx
    print(f"skyseg.onnx  {SKYSEG.stat().st_size/1e6:.0f} MB")

## 6 · The runner

Shells out to `recon/reconstruct.py` so the code path is byte-identical to the local one. Output
is teed to a log because a truncated notebook cell is not evidence — and because piping this to
anything that swallows the exit code is exactly how an OOM-killed run gets mistaken for a success.

In [ ]:
import json, pathlib, subprocess, sys, time

WORKP = pathlib.Path(WORK); (WORKP / "runs").mkdir(parents=True, exist_ok=True)
(WORKP / "logs").mkdir(parents=True, exist_ok=True)


def frames_dir(scene):
    d = pathlib.Path(LINGBOT_SRC, "example", scene)
    if d.is_dir():
        return d
    d = WORKP / "frames" / scene           # extracted from one of VIDEOS
    assert d.is_dir(), f"no frames for {scene!r}"
    return d


def reconstruct(scene, tag=None, quiet=False, **over):
    """Run one reconstruction. `over` overrides UPSTREAM. Returns run.json, or None on failure."""
    cfg = dict(UPSTREAM); cfg.update(over)
    tag = tag or scene
    out = WORKP / "runs" / tag
    log = WORKP / "logs" / f"{tag}.log"

    if (out / "run.json").exists():
        print(f"[{tag}] already done, reusing")
        return json.loads((out / "run.json").read_text())

    cmd = [sys.executable, str(RECON / "reconstruct.py"),
           "--frames", str(frames_dir(scene)), "--out", str(out),
           "--model_path", str(CKPT),
           "--conf_percentile", str(CONF_PERCENTILE),
           "--pixel_stride", str(PIXEL_STRIDE),
           "--vram_fraction", str(VRAM_FRACTION)]
    for k, v in cfg.items():
        cmd += [f"--{k}", str(v)]
    if MASK_SKY.get(scene) and SKYSEG.exists():
        cmd += ["--mask_sky", "--skyseg_model", str(SKYSEG)]

    print(f"[{tag}] kvsw={cfg['kv_cache_sliding_window']} kfi={cfg['keyframe_interval']} "
          f"mode={cfg['mode']}" + ("  +sky" if MASK_SKY.get(scene) else ""))
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env={**os.environ, "LINGBOT_SRC": LINGBOT_SRC})
        for line in p.stdout:
            fh.write(line)
            # Two upstream lines are expected every run and are not warnings:
            # "flashinfer not available" and "Failed to load pretrained weights: [Errno 2]".
            if not quiet:
                sys.stdout.write(line)
        rc = p.wait()

    if rc != 0:
        print(f"[{tag}] FAILED rc={rc} -- tail of {log}:")
        print("".join(log.read_text().splitlines(keepends=True)[-15:]))
        return None

    rec = json.loads((out / "run.json").read_text())
    print(f"[{tag}] {time.time()-t0:.0f}s  {rec['n_points']:,} pts  "
          f"{rec['peak_vram_gb']:.2f} GB  ratio {rec['traj_length_over_extent']}")
    return rec


# Optional: extract frames from your own video files before the demo scenes run.
for name, path in VIDEOS.items():
    d = WORKP / "frames" / name
    if not d.is_dir():
        subprocess.run([sys.executable, str(RECON / "extract_frames.py"), path, str(d),
                        "--fps", str(VIDEO_FPS)], check=True)
    if name not in SCENES:
        SCENES.append(name)

print("runner ready")

## 7 · Both demo scenes at upstream's settings

The number to watch is `traj_length_over_extent` — camera path length divided by scene size. It is
a drift **detector**, not a quality score (a 2026-08-05 run scored 2.87 while looking terrible), but
a value in the twenties is unambiguous: the poses have collapsed.

| scene | local (this repo, 8 GB) | upstream's published demo |
| --- | --- | --- |
| loop | **3.36** at kvsw 24 — healthy | coherent |
| courthouse | **24.9** at kvsw 16 — collapsed | coherent |

In [ ]:
LOCAL = {"loop": dict(ratio=3.36, kvsw=24, vram=7.19, pts=4_073_506),
         "courthouse": dict(ratio=24.88, kvsw=16, vram=None, pts=None)}

runs = {}
for scene in SCENES:
    runs[scene] = reconstruct(scene, tag=f"{scene}_upstream", quiet=True)

print()
for scene, r in runs.items():
    if r is None:
        print(f"{scene:12s} FAILED"); continue
    base = LOCAL.get(scene, {}).get("ratio")
    delta = f"   local was {base} at kvsw {LOCAL[scene]['kvsw']}" if base else ""
    print(f"{scene:12s} ratio {r['traj_length_over_extent']:6.2f}   "
          f"{r['n_points']:>10,} pts   {r['peak_vram_gb']:5.2f} GB   "
          f"{r['fps']:.2f} fps{delta}")

## 8 · Where is the cache cliff?

Locally this is unanswerable: 24 is the ceiling, so the ladder has one rung. Here the whole ladder
fits.

The pair to read carefully is **`kvsw 16 / kfi 2`** against **`kvsw 32 / kfi 1`**. Both cover the
same span of footage; the first does it with 16 cached views, the second with 32. If only the view
count matters, `(32, 1)` wins clearly and VRAM is the entire story — which makes the cloud GPU a
correctness requirement for Phase 7, not a throughput nicety. If they land together, the model
needs *horizon*, and buying VRAM alone will not fix the expedition's long walks.

In [ ]:
sweep_rows = []
if RUN_SWEEP:
    for kvsw, kfi in SWEEP:
        if kvsw > 64 and VRAM_GB < 20:
            print(f"skipping kvsw {kvsw} on a {VRAM_GB:.0f} GB card"); continue
        tag = f"{SWEEP_SCENE}_kv{kvsw}_kfi{kfi}"
        # (64, 1) is the upstream run from cell 7 -- reuse it rather than paying for it twice.
        if (kvsw, kfi) == (UPSTREAM["kv_cache_sliding_window"], UPSTREAM["keyframe_interval"]):
            r = runs.get(SWEEP_SCENE)
        else:
            r = reconstruct(SWEEP_SCENE, tag=tag, quiet=True,
                            kv_cache_sliding_window=kvsw, keyframe_interval=kfi)
        sweep_rows.append((kvsw, kfi, r))

    print(f"\n{SWEEP_SCENE}: cache sweep")
    print(f"{'kvsw':>5} {'kfi':>4} {'ratio':>8} {'peak GB':>8} {'fps':>6} {'points':>11}")
    for kvsw, kfi, r in sweep_rows:
        if r is None:
            print(f"{kvsw:>5} {kfi:>4} {'OOM/FAIL':>8}"); continue
        print(f"{kvsw:>5} {kfi:>4} {r['traj_length_over_extent']:>8.2f} "
              f"{r['peak_vram_gb']:>8.2f} {r['fps']:>6.2f} {r['n_points']:>11,}")

## 9 · Verdict — is compute the bottleneck?

Decided from the numbers, not from how the renders feel. Thresholds match the repo's:
`traj_length_over_extent` ≤ 6 is the bar `calibrate_scale.py` requires before it will trust poses
enough to anchor scale; ≤ 4 is comfortably healthy.

In [ ]:
VERDICT = {}
ch = runs.get("courthouse")
lo = runs.get("loop")

if ch:
    r64 = ch["traj_length_over_extent"]
    r16 = next((r["traj_length_over_extent"] for k, f, r in sweep_rows
                if (k, f) == (16, 1) and r), None)
    print(f"courthouse @ upstream kvsw 64 : {r64:.2f}")
    if r16:
        print(f"courthouse @ our kvsw 16      : {r16:.2f}   (local run on 8 GB: 24.88)")

    if r64 <= 6:
        VERDICT["courthouse"] = "COMPUTE-BOUND"
        print("\n=> COMPUTE WAS THE BOTTLENECK. Upstream's config reconstructs courthouse "
              "correctly; the 8 GB card could not hold the cache it needs.")
        print("   Phase 3 inference should move to a rented GPU, and the ~25 s-per-scene "
              "limit in notes/open-questions.md is a property of our hardware, not the model.")
    elif r16 and abs(r64 - r16) / max(r16, 1e-9) < 0.15:
        VERDICT["courthouse"] = "NOT-COMPUTE"
        print("\n=> NOT COMPUTE. A 4x cache changed nothing, so the collapse is not cache size. "
              "Suspect the footage or the preprocessing path, not the card.")
    else:
        VERDICT["courthouse"] = "PARTIAL"
        print(f"\n=> PARTIAL. More cache helps ({r16:.1f} -> {r64:.1f}) but does not reach the "
              "ratio<=6 bar that calibrate_scale.py requires. Cache is one term, not the term.")

if lo:
    print(f"\nloop @ upstream kvsw 64 : {lo['traj_length_over_extent']:.2f}   "
          f"(local at kvsw 24: 3.36)")
    print("  loop is the control: it was already healthy locally, so a large change here would "
          "mean the two setups differ in some way other than cache size.")

## 10 · Heavy Open3D cleanup → a metric, ground-aligned map

Two scripts, unmodified from the repo:

- **`calibrate_scale.py`** — fits candidate ground planes, keeps the one that holds camera height
  *constant* (inlier count picks a wall in a corridor), then divides median camera height by an
  assumed 1.5 m eye height. It **refuses** above drift ratio 6, because poses that drifted cannot
  anchor anything. If courthouse now passes that gate, that alone is a result.
- **`clean_cloud.py --scale auto`** — scales to metres *first* so every filter size below is a real
  distance, then statistical outlier removal → radius filtering (the flying-pixel streaks shed at
  occlusion edges, which statistical removal misses because each streak is locally dense along its
  own filament) → 2 cm voxel downsample → ground plane to +Z at z=0.

`CLEAN_HEAVY` tightens `--std-ratio` to 1.5 and `--min-neighbors` to 16.

In [ ]:
import json, subprocess, sys

def clean(scene, tag=None):
    tag = tag or f"{scene}_upstream"
    d = WORKP / "runs" / tag
    if not (d / "cloud.ply").exists():
        print(f"[{tag}] no cloud"); return None

    cal = subprocess.run([sys.executable, str(RECON / "calibrate_scale.py"), str(d),
                          "--camera-height", str(CAMERA_HEIGHT_M)],
                         capture_output=True, text=True)
    print(f"── {tag}: scale")
    print(cal.stdout.strip())
    if cal.returncode != 0:
        # Almost always the drift guard: "refusing to anchor scale, ratio N > 6".
        print("  REFUSED:", cal.stderr.strip().splitlines()[-1] if cal.stderr.strip() else f"rc={cal.returncode}")

    cmd = [sys.executable, str(RECON / "clean_cloud.py"), str(d)]
    if (d / "scale.json").exists():
        cmd += ["--scale", "auto"]
    else:
        # No trustworthy poses -> no metric anchor. The cloud still cleans and still
        # renders; it just stays in arbitrary units and cannot become terrain.
        print("  no scale.json -> cleaning in arbitrary units (NOT terrain-ready)")
    if CLEAN_HEAVY:
        cmd += ["--std-ratio", "1.5", "--min-neighbors", "16"]

    cl = subprocess.run(cmd, capture_output=True, text=True)
    print(f"── {tag}: clean"); print(cl.stdout.strip() or cl.stderr.strip())
    return json.loads((d / "clean_stats.json").read_text()) if cl.returncode == 0 else None


cleaned = {s: clean(s) for s in SCENES if runs.get(s)}

## 11 · Look at the maps

Tries the repo's `inspect_cloud.py` first — it renders four orbit views through Open3D's headless
EGL path and writes `inspect_stats.json` (point count, nearest-neighbour spacing, pose-jump flag).
Colab does not always expose EGL to Open3D, so there is a matplotlib fallback that plots the same
four views from the cleaned cloud's own colours.

In [ ]:
import numpy as np, subprocess, sys
import matplotlib.pyplot as plt
from IPython.display import Image, display


def mpl_preview(ply, title, max_pts=250_000):
    pcd = o3d.io.read_point_cloud(str(ply))
    p = np.asarray(pcd.points)
    c = np.asarray(pcd.colors)
    if len(p) > max_pts:
        i = np.random.default_rng(0).choice(len(p), max_pts, replace=False)
        p, c = p[i], (c[i] if len(c) else None)
    c = c if len(c) else np.full((len(p), 3), 0.35)

    th = np.deg2rad(45)
    rot = p @ np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]]).T
    views = [("top  (x,y)", p[:, 0], p[:, 1]), ("front (x,z)", p[:, 0], p[:, 2]),
             ("side  (y,z)", p[:, 1], p[:, 2]), ("oblique", rot[:, 0], rot[:, 2])]

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for ax, (name, u, v) in zip(axes.ravel(), views):
        ax.scatter(u, v, c=np.clip(c, 0, 1), s=0.06, linewidths=0)
        ax.set_aspect("equal"); ax.set_title(name, fontsize=10)
        ax.set_facecolor("white"); ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(); plt.show()


for scene in SCENES:
    tag = f"{scene}_upstream"
    d = WORKP / "runs" / tag
    ply = d / "cloud_clean.ply"
    if not ply.exists():
        continue
    st = json.loads((d / "clean_stats.json").read_text())
    print(f"\n{'='*70}\n{scene}: {st['n_out']:,} points, extent "
          f"{st['extent_out']} {st['units']}, {st['removed_pct']}% removed\n{'='*70}")

    shown = False
    try:
        r = subprocess.run([sys.executable, str(RECON / "inspect_cloud.py"), str(d),
                            "--clean", "--tag", "clean_"], capture_output=True, text=True, timeout=900)
        pngs = sorted(d.glob("view_clean_*.png"))
        if r.returncode == 0 and pngs:
            for q in pngs:
                display(Image(filename=str(q), width=760))
            shown = True
    except Exception as e:
        print("inspect_cloud unavailable:", type(e).__name__)
    if not shown:
        print("(Open3D offscreen EGL unavailable here -- matplotlib fallback)")
        mpl_preview(ply, f"{scene} — cleaned")

## 12 · Take the results home

Zips the per-scene artifacts and prints `notes/experiments.md` rows to paste. Rows are mandatory
per `CLAUDE.md` — every reconstruction run gets one, including the failures, which is the point
of an append-only log.

Fill in `commit` with the short hash of the code that ran (`git rev-parse --short HEAD` on the
machine you zipped `recon/` from).

In [ ]:
import shutil, zipfile, datetime

STAMP = datetime.date.today().isoformat()
bundle = pathlib.Path(f"/content/lingbotmap_colab_{STAMP}.zip")

KEEP = ["run.json", "scale.json", "clean_stats.json", "inspect_stats.json",
        "cloud_clean.ply", "trajectory.npz"]
if KEEP_RAW_PLY:
    KEEP.append("cloud.ply")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for d in sorted((WORKP / "runs").iterdir()):
        for n in KEEP:
            if (d / n).exists():
                z.write(d / n, f"{d.name}/{n}")
        for q in d.glob("view_*.png"):
            z.write(q, f"{d.name}/{q.name}")
    for q in (WORKP / "logs").glob("*.log"):
        z.write(q, f"logs/{q.name}")

print(f"{bundle}  {bundle.stat().st_size/1e6:.1f} MB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/GeologicDome/colab_runs"); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(bundle, dst); print("copied to", dst)

# ── experiments.md rows ─────────────────────────────────────────────────────
gpu = P.name.replace("NVIDIA ", "")
print("\n" + "="*70 + "\npaste into notes/experiments.md:\n")
for tag, r in [(t, json.loads((WORKP / 'runs' / t / 'run.json').read_text()))
               for t in sorted(p.name for p in (WORKP / "runs").iterdir())
               if (WORKP / "runs" / t / "run.json").exists()]:
    cfg = (f"LingBot-Map base on **Colab {gpu} {VRAM_GB:.0f} GB**, upstream `example/"
           f"{tag.split('_')[0]}` ({r['n_frames']} frames), streaming, "
           f"**kvsw={r['kv_cache_sliding_window']}** kfi={r['keyframe_interval']}, "
           f"nsf={r['num_scale_frames']}, 518 crop, conf p{r['conf_percentile']:.0f}")
    met = (f"{r['inference_s']:.0f} s, {r['fps']:.2f} fps, peak VRAM {r['peak_vram_gb']:.2f} GB, "
           f"{r['n_points']:,} pts, **ratio {r['traj_length_over_extent']}**")
    print(f"| {STAMP}-colab-{tag.replace('_','-')} | <hash> | {cfg} | — | {met} | <takeaway> |")

from google.colab import files
files.download(str(bundle))